<a href="https://colab.research.google.com/github/prakhar-2408/Flexe-Glove-EClub/blob/main/Week_4/Sign_Language/Sensor_test/Sign_language_lstm_gru_model_240763.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import io
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

df = pd.read_csv("SignSpeakDataset.csv")

df = df.drop(columns=['_id'])

y = df['word']
X = df.drop(columns=['word'])
for col in X.columns:
  X[col] = X[col].fillna(0)

# Handle missing values in target (dropping rows with missing target)
if y.isnull().sum() > 0:
    print("\nMissing values detected in target. Dropping rows with missing target.")
    original_rows = df.shape[0] # Note: df here already had '_id' dropped if it existed
    # Create a temporary dataframe to drop NaNs and then re-assign X and y
    temp_df = pd.concat([X, y], axis=1).dropna(subset=[y.name])
    if temp_df.shape[0] < original_rows:
        rows_dropped = original_rows - temp_df.shape[0]
        print(f"Dropped {rows_dropped} rows due to missing target values.")
        X = temp_df.iloc[:, :-1]
        y = temp_df.iloc[:, -1]
    else:
        print("No rows dropped for missing target after re-checking.")


label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
print(f"Original word classes: {label_encoder.classes_}")
print(f"Number of classes: {num_classes}")


scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X)


n_features = X_scaled.shape[1]
print(f"\nNumber of features per timestep: {n_features}")

# Define n_steps: the number of timesteps (frames) in each sequence.
# This is a critical hyperparameter. Adjust it based on the nature of your data.
# For sign language, a sign might span many frames.
n_steps = 20 # Example: 20 timesteps per sequence.

print(f"Creating sequences with {n_steps} timesteps per sequence.")

def create_sequences(X_data, y_data, n_steps):
    """
    Creates sequences from flat data suitable for LSTM.
    Each sequence will have 'n_steps' timesteps.
    The target for a sequence is the target of its last timestep.
    """
    Xs, ys = [], []
    if len(X_data) <= n_steps:
          raise ValueError(f"Not enough data to create sequences with n_steps={n_steps}. Dataset has {len(X_data)} rows.")

    for i in range(len(X_data) - n_steps):
        # Extract a sequence of n_steps from the input data
        seq = X_data[i:(i + n_steps)]
        # Assign the label of the last timestep in the sequence as the sequence's label
        label = y_data[i + n_steps - 1]
        Xs.append(seq)
        ys.append(label)
    return np.array(Xs), np.array(ys)

X_sequences, y_sequences = create_sequences(X_scaled, y_encoded, n_steps)

print(f"Shape of X_sequences (samples, timesteps, features): {X_sequences.shape}")
print(f"Shape of y_sequences (samples,): {y_sequences.shape}")


X_train, X_test, y_train, y_test = train_test_split(
    X_sequences, y_sequences, test_size=0.2, random_state=42, stratify=y_sequences
)
print(f"\nTraining set shape: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Testing set shape: X_test {X_test.shape}, y_test {y_test.shape}")

# Step 4: Build LSTM Model
model = Sequential()
model.add(LSTM(units=50, activation='relu', input_shape=(n_steps, n_features), return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(units=50, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(units=num_classes, activation='softmax'))

model.summary()

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Step 5: Train Model
early_stopping = EarlyStopping(patience=5, restore_best_weights=True)
print("\nStarting model training...")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
    callbacks=[early_stopping]
)


print("\nEvaluating model on the test set...")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

print("\nTraining History (first 5 epochs):")
print(pd.DataFrame(history.history).head())

# Step 7: Make predictions on test data
print("\nMaking predictions on the test set...")
y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

print("\n--- Sample Predictions ---")
num_samples_to_show = 20
print(f"Showing first {num_samples_to_show} predictions vs. actual labels:")
for i in range(min(num_samples_to_show, len(y_test))):
    actual_label_encoded = y_test[i]
    predicted_label_encoded = y_pred_classes[i]

    actual_label_word = label_encoder.inverse_transform([actual_label_encoded])[0]
    predicted_label_word = label_encoder.inverse_transform([predicted_label_encoded])[0]

    print(f"Sample {i+1}: Actual Word: '{actual_label_word}' (Encoded: {actual_label_encoded}), Predicted Word: '{predicted_label_word}' (Encoded: {predicted_label_encoded})")
print("--------------------------")

Original word classes: ['1' '10' '2' '3' '4' '5' '6' '7' '8' '9' 'a' 'b' 'c' 'd' 'e' 'f' 'g' 'h'
 'i' 'j' 'k' 'l' 'm' 'n' 'o' 'p' 'q' 'r' 's' 't' 'u' 'v' 'w' 'x' 'y' 'z']
Number of classes: 36

Number of features per timestep: 395
Creating sequences with 20 timesteps per sequence.
Shape of X_sequences (samples, timesteps, features): (7180, 20, 395)
Shape of y_sequences (samples,): (7180,)

Training set shape: X_train (5744, 20, 395), y_train (5744,)
Testing set shape: X_test (1436, 20, 395), y_test (1436,)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_28 (LSTM)                       │ (None, 20, 50)              │          89,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_30 (Dropout)                 │ (None, 20, 50)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_29 (LSTM)                       │ (None, 50)                  │          20,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_31 (Dropout)                 │ (None, 50)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_25 (Dense)                     │ (None, 36)                  │           1,836 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 111,236 (434.52 KB)

 Trainable params: 111,236 (434.52 KB)

 Non-trainable params: 0 (0.00 B)


Starting model training...
Epoch 1/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.1543 - loss: 3.0649 - val_accuracy: 0.5113 - val_loss: 1.5263
Epoch 2/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0.5151 - loss: 1.5261 - val_accuracy: 0.6974 - val_loss: 0.8110
Epoch 3/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 6s 34ms/step - accuracy: 0.6577 - loss: 1.0073 - val_accuracy: 0.8052 - val_loss: 0.6011
Epoch 4/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 11s 41ms/step - accuracy: 0.7561 - loss: 0.7009 - val_accuracy: 0.8330 - val_loss: 0.5323
Epoch 5/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.7931 - loss: 0.5761 - val_accuracy: 0.8574 - val_loss: 0.4012
Epoch 6/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.8358 - loss: 0.4442 - val_accuracy: 0.8678 - val_loss: 0.4047
Epoch 7/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 13s 60ms/step - accuracy: 0.8405 - loss: 0.4854 - val_accuracy: 0.8974 - val_loss: 0.2778
Epoch 8/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 5s 30ms/step - accuracy: 0

In [ ]:
import pandas as pd
import numpy as np
import io
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

df = pd.read_csv("SignSpeakDataset.csv")

df = df.drop(columns=['_id'])

y = df['word']
X = df.drop(columns=['word'])
for col in X.columns:
  X[col] = X[col].fillna(0)

# Handle missing values in target (dropping rows with missing target)
if y.isnull().sum() > 0:
    print("\nMissing values detected in target. Dropping rows with missing target.")
    original_rows = df.shape[0] # Note: df here already had '_id' dropped if it existed
    # Create a temporary dataframe to drop NaNs and then re-assign X and y
    temp_df = pd.concat([X, y], axis=1).dropna(subset=[y.name])
    if temp_df.shape[0] < original_rows:
        rows_dropped = original_rows - temp_df.shape[0]
        print(f"Dropped {rows_dropped} rows due to missing target values.")
        X = temp_df.iloc[:, :-1]
        y = temp_df.iloc[:, -1]
    else:
        print("No rows dropped for missing target after re-checking.")


label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)
print(f"Original word classes: {label_encoder.classes_}")
print(f"Number of classes: {num_classes}")


scaler = MinMaxScaler(feature_range=(0, 1))
X_scaled = scaler.fit_transform(X)


n_features = X_scaled.shape[1]
print(f"\nNumber of features per timestep: {n_features}")

# Define n_steps: the number of timesteps (frames) in each sequence.
# This is a critical hyperparameter. Adjust it based on the nature of your data.
# For sign language, a sign might span many frames.
n_steps = 20 # Example: 20 timesteps per sequence.

print(f"Creating sequences with {n_steps} timesteps per sequence.")

def create_sequences(X_data, y_data, n_steps):
    """
    Creates sequences from flat data suitable for LSTM.
    Each sequence will have 'n_steps' timesteps.
    The target for a sequence is the target of its last timestep.
    """
    Xs, ys = [], []
    if len(X_data) <= n_steps:
          raise ValueError(f"Not enough data to create sequences with n_steps={n_steps}. Dataset has {len(X_data)} rows.")

    for i in range(len(X_data) - n_steps):
        # Extract a sequence of n_steps from the input data
        seq = X_data[i:(i + n_steps)]
        # Assign the label of the last timestep in the sequence as the sequence's label
        label = y_data[i + n_steps - 1]
        Xs.append(seq)
        ys.append(label)
    return np.array(Xs), np.array(ys)

X_sequences, y_sequences = create_sequences(X_scaled, y_encoded, n_steps)

print(f"Shape of X_sequences (samples, timesteps, features): {X_sequences.shape}")
print(f"Shape of y_sequences (samples,): {y_sequences.shape}")


X_train, X_test, y_train, y_test = train_test_split(
    X_sequences, y_sequences, test_size=0.2, random_state=42, stratify=y_sequences
)
print(f"\nTraining set shape: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Testing set shape: X_test {X_test.shape}, y_test {y_test.shape}")

# Step 4: Build LSTM Model
model = Sequential()
model.add(GRU(units=50, activation='relu', input_shape=(n_steps, n_features), return_sequences=True))
model.add(Dropout(0.2))
model.add(GRU(units=50, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(units=num_classes, activation='softmax'))

model.summary()

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Step 5: Train Model
early_stopping = EarlyStopping(patience=5, restore_best_weights=True)
print("\nStarting model training...")
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
    callbacks=[early_stopping]
)


print("\nEvaluating model on the test set...")
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

print("\nTraining History (first 5 epochs):")
print(pd.DataFrame(history.history).head())

# Step 7: Make predictions on test data
print("\nMaking predictions on the test set...")
y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

print("\n--- Sample Predictions ---")
num_samples_to_show = 20
print(f"Showing first {num_samples_to_show} predictions vs. actual labels:")
for i in range(min(num_samples_to_show, len(y_test))):
    actual_label_encoded = y_test[i]
    predicted_label_encoded = y_pred_classes[i]

    actual_label_word = label_encoder.inverse_transform([actual_label_encoded])[0]
    predicted_label_word = label_encoder.inverse_transform([predicted_label_encoded])[0]

    print(f"Sample {i+1}: Actual Word: '{actual_label_word}' (Encoded: {actual_label_encoded}), Predicted Word: '{predicted_label_word}' (Encoded: {predicted_label_encoded})")
print("--------------------------")

Original word classes: ['1' '10' '2' '3' '4' '5' '6' '7' '8' '9' 'a' 'b' 'c' 'd' 'e' 'f' 'g' 'h'
 'i' 'j' 'k' 'l' 'm' 'n' 'o' 'p' 'q' 'r' 's' 't' 'u' 'v' 'w' 'x' 'y' 'z']
Number of classes: 36

Number of features per timestep: 395
Creating sequences with 20 timesteps per sequence.
Shape of X_sequences (samples, timesteps, features): (7180, 20, 395)
Shape of y_sequences (samples,): (7180,)

Training set shape: X_train (5744, 20, 395), y_train (5744,)
Testing set shape: X_test (1436, 20, 395), y_test (1436,)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ gru_2 (GRU)                          │ (None, 20, 50)              │          67,050 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_32 (Dropout)                 │ (None, 20, 50)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru_3 (GRU)                          │ (None, 50)                  │          15,300 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_33 (Dropout)                 │ (None, 50)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_26 (Dense)                     │ (None, 36)                  │           1,836 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 84,186 (328.85 KB)

 Trainable params: 84,186 (328.85 KB)

 Non-trainable params: 0 (0.00 B)


Starting model training...
Epoch 1/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 11s 38ms/step - accuracy: 0.1659 - loss: 2.9692 - val_accuracy: 0.5757 - val_loss: 1.2236
Epoch 2/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.5511 - loss: 1.2675 - val_accuracy: 0.7443 - val_loss: 0.6935
Epoch 3/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 11s 42ms/step - accuracy: 0.6786 - loss: 0.8660 - val_accuracy: 0.7878 - val_loss: 0.5321
Epoch 4/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 10s 41ms/step - accuracy: 0.7726 - loss: 0.6054 - val_accuracy: 0.8417 - val_loss: 0.3843
Epoch 5/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 6s 34ms/step - accuracy: 0.8229 - loss: 0.4811 - val_accuracy: 0.8991 - val_loss: 0.3061
Epoch 6/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.8650 - loss: 0.3767 - val_accuracy: 0.8922 - val_loss: 0.2752
Epoch 7/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.8713 - loss: 0.3357 - val_accuracy: 0.9304 - val_loss: 0.1600
Epoch 8/50
162/162 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: